In [1]:
from model.setup import setup_scenario


platform, msg, path = setup_scenario('netzero')

2024-09-17 16:21:40,454  INFO at.ac.iiasa.ixmp.Platform:165 - Welcome to the IX modeling platform!
2024-09-17 16:21:40,460  INFO at.ac.iiasa.ixmp.Platform:166 -  connected to database 'jdbc:hsqldb:file:/Users/muhammadawais/.local/share/ixmp/localdb/default' (user: ixmp)...
Sheet name: type_tec
sets type_tec
Sheet name: cat_tec
sets cat_tec
Sheet name: shares
sets shares
Sheet name: map_shares_commodity_share
sets map_shares_commodity_share
Sheet name: map_shares_commodity_total
sets map_shares_commodity_total
Sheet name: share_commodity_lo
par share_commodity_lo
['share_h2_rc' 'share_h2_i' 'share_h2_fc_i' 'share_h2_fc_RC'
 'share_h2_trp']


In [5]:
df = msg.var('EMISS')

In [ ]:
nodes = 'World'

In [6]:
df_emis = df[(df['emission'] == 'TCE') & (df['type_tec'] == 'all') & ~(df['node'].str.contains('World'))]

In [15]:
df_emis.to_csv('emissions_baseline.csv')

In [7]:
import pandas as pd

In [8]:
df_emis = pd.read_csv('emissions_baseline.csv')

In [9]:
import pandas as pd


emissions_df = df_emis

# Define the exact emissions trajectory for Canada (in MtCO2eq)
canada_trajectory = {
    2020: 659e6,   # 659 MtCO2eq
    2025: 549e6,   # 549 MtCO2eq
    2030: 439e6,   # 439 MtCO2eq
    2035: 329e6,   # 329 MtCO2eq
    2040: 220e6,   # 220 MtCO2eq
    2045: 110e6,   # 110 MtCO2eq
    2050: 50e6,    # 50,000 MtCO2eq (likely meant to be 50 MtCO2eq)
    2055: 40e6,    # 40,000 MtCO2eq (likely meant to be 40 MtCO2eq)
    2060: 30e6     # 30,000 MtCO2eq (likely meant to be 30 MtCO2eq)
}

# Extract the unique provinces (nodes) excluding Canada
provinces = emissions_df['node'].unique()
provinces = provinces[provinces != 'Canada']

# Initialize a list to store the emission trajectories for each province
emission_trajectories_list = []

# Iterate over each province and apply the reduction schedule
for province in provinces:
    # Extract the emission data for the province
    province_data = emissions_df[(emissions_df['node'] == province)]
    
    # Create a dictionary to store the year and emission values
    province_emissions = {'province': [], 'year': [], 'emission': []}
    
    # Get the 2020 base year emission for the province
    base_year_emission = province_data[province_data['year'] == 2020]['lvl'].values[0]
    
    # Get the total emissions for Canada in 2020
    canada_2020_emission = canada_trajectory[2020]
    
    # Calculate the proportion of this province's emissions relative to Canada's total in 2020
    province_proportion = base_year_emission / canada_2020_emission
    
    # Iterate over the Canada trajectory and calculate the emissions for this province
    for year, canada_target in canada_trajectory.items():
        adjusted_emission = province_proportion * canada_target
        
        # Store the emission value in the dictionary
        province_emissions['province'].append(province)
        province_emissions['year'].append(year)
        province_emissions['emission'].append(adjusted_emission)
    
    # Convert the dictionary to a DataFrame and append to the list
    province_emissions_df = pd.DataFrame(province_emissions)
    emission_trajectories_list.append(province_emissions_df)

# Concatenate all the DataFrames into a single DataFrame
emission_trajectories_df = pd.concat(emission_trajectories_list).reset_index(drop=True)

# Save the resulting DataFrame to a new CSV file


# Display the head of the resulting DataFrame
emission_trajectories_df.head()


,province,year,emission
0,Alberta,2020,9.614798e+07
1,Alberta,2025,8.009900e+07
2,Alberta,2030,6.405002e+07
3,Alberta,2035,4.800104e+07
4,Alberta,2040,3.209796e+07


In [10]:
emission_trajectories_df

,province,year,emission
0,Alberta,2020,9.614798e+07
1,Alberta,2025,8.009900e+07
2,Alberta,2030,6.405002e+07
3,Alberta,2035,4.800104e+07
4,Alberta,2040,3.209796e+07
...,...,...,...
112,Yukon,2040,3.197597e+05
113,Yukon,2045,1.598799e+05
114,Yukon,2050,7.267266e+04
115,Yukon,2055,5.813813e+04


In [11]:
df_bound = emission_trajectories_df.copy()
df_bound.rename(columns={'province':'node','year':'type_year','emission':'value'}, inplace=True)

In [12]:
df_bound['type_emission'] = 'TCE'
df_bound['type_tec'] = 'all'
df_bound['unit'] = '-'

In [64]:
df_bound

,node,type_year,value,type_emission,type_tec,unit
0,Alberta,2020,9.614798e+07,TCE,all,-
1,Alberta,2025,8.009900e+07,TCE,all,-
2,Alberta,2030,6.405002e+07,TCE,all,-
3,Alberta,2035,4.800104e+07,TCE,all,-
4,Alberta,2040,3.209796e+07,TCE,all,-
...,...,...,...,...,...,...
112,Yukon,2040,3.197597e+05,TCE,all,-
113,Yukon,2045,1.598799e+05,TCE,all,-
114,Yukon,2050,7.267266e+04,TCE,all,-
115,Yukon,2055,5.813813e+04,TCE,all,-


In [74]:
df_bound = df_bound[df_bound['type_year'] > 2020]

In [80]:
msg.commit('')

In [81]:
# msg.remove_solution()
msg.check_out()
# msg.add_par('demand', df)

In [82]:
msg.remove_par('bound_emission', df_bound)

In [84]:
df_bound['value'] = df_bound['value'] / 4

In [85]:
msg.add_par('bound_emission', df_bound)

In [39]:

df = msg.par('growth_activity_lo')

In [40]:
tecs = ['loil_trp', 'loil_fs', 'loil_i', 'loil_rc', 'foil_fs','oil_extr_6','gas_fs']
df_selected = df[df['technology'].isin(tecs)]
df_selected.loc[df_selected['technology'].isin(tecs), 'value'] *= 2

In [78]:
msg.check_out()

In [30]:
msg.add_par('growth_activity_lo', df_selected)

In [17]:
msg.remove_solution()

In [20]:
path

PosixPath('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/netzero')

In [61]:
import pandas as pd

df = pd.read_csv(path/'tax_emission.csv')

In [ ]:
df_bound

,node,type_emission,type_tec,year,value
0,Alberta,TCE,all,2025,59.2
1,Alberta,TCE,all,2030,103.6
2,Alberta,TCE,all,2035,125.8
3,Alberta,TCE,all,2040,125.8
4,Alberta,TCE,all,2045,125.8
...,...,...,...,...,...
99,Yukon,TCE,all,2040,125.8
100,Yukon,TCE,all,2045,125.8
101,Yukon,TCE,all,2050,125.8
102,Yukon,TCE,all,2055,125.8


In [63]:
df['type_year'] = df['year']

In [29]:
msg.add_par('tax_emission', df)

In [30]:
msg.remove_par('bound_emission',df)

In [86]:
msg.commit('update tax and emission bound')

In [87]:
caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version)
msg.solve(solve_options={'barcrossalg': '2','lpmethod':'1'}, case=caseName)

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 09/10/24 15:17:34 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225a/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__netzero__v6.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__netzero__v6.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__netzero__v6.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
        

ModelError: GAMS errored with return code 3:
    There was an execution error

For details, see the terminal output above, plus:
Listing   : /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.lst
Log file  : /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.log
Input data: /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__netzero__v6.gdx

In [51]:
df = msg.var("DEMAND").rename(columns={"lvl": "value"}).assign(unit="GWa")

In [46]:
df

,node,commodity,level,year,time,value,mrg,unit
0,Alberta,transport,useful,2020,year,13.134694,0.0,GWa
1,Alberta,transport,useful,2025,year,16.032357,0.0,GWa
2,Alberta,transport,useful,2030,year,17.398884,0.0,GWa
3,Alberta,transport,useful,2035,year,19.330618,0.0,GWa
4,Alberta,transport,useful,2040,year,21.379482,0.0,GWa
...,...,...,...,...,...,...,...,...
697,Yukon,rc_therm,useful,2040,year,0.067450,0.0,GWa
698,Yukon,rc_therm,useful,2045,year,0.076771,0.0,GWa
699,Yukon,rc_therm,useful,2050,year,0.086775,0.0,GWa
700,Yukon,rc_therm,useful,2055,year,0.097813,0.0,GWa


In [36]:
df.to_csv('demands_new.csv')

In [52]:
df_new = msg.par("demand")

In [59]:
df_new[(df_new['node'] =='Alberta') & (df['commodity'] == 'transport')]

,node,commodity,level,year,time,value,unit
540,Alberta,rc_therm,useful,2020,year,14.672228,GWa
541,Alberta,rc_therm,useful,2025,year,17.405320,GWa
542,Alberta,rc_therm,useful,2030,year,20.515764,GWa
543,Alberta,rc_therm,useful,2035,year,23.531199,GWa
544,Alberta,rc_therm,useful,2040,year,26.867014,GWa
545,Alberta,rc_therm,useful,2045,year,30.579487,GWa
546,Alberta,rc_therm,useful,2050,year,34.564266,GWa
547,Alberta,rc_therm,useful,2055,year,38.961146,GWa
548,Alberta,rc_therm,useful,2060,year,43.550800,GWa


In [60]:
df[(df['node'] =='Alberta') & (df['commodity'] == 'transport')]

,node,commodity,level,year,time,value,mrg,unit
0,Alberta,transport,useful,2020,year,13.134694,0.0,GWa
1,Alberta,transport,useful,2025,year,16.032357,0.0,GWa
2,Alberta,transport,useful,2030,year,17.398884,0.0,GWa
3,Alberta,transport,useful,2035,year,19.330618,0.0,GWa
4,Alberta,transport,useful,2040,year,21.379482,0.0,GWa
5,Alberta,transport,useful,2045,year,23.795802,0.0,GWa
6,Alberta,transport,useful,2050,year,26.416787,0.0,GWa
7,Alberta,transport,useful,2055,year,29.254100,0.0,GWa
8,Alberta,transport,useful,2060,year,32.134819,0.0,GWa


In [53]:
is_different = not df_new.equals(df[['node', 'commodity', 'level', 'year', 'time', 'value', 'unit']])

In [54]:
is_different

True

In [43]:
df_new.equals(df[['node', 'commodity', 'level', 'year', 'time', 'value', 'unit']])

False

In [41]:
differences

,node,commodity,level,year,time,value,unit


In [39]:
is_different

True

In [ ]:

scen.check_out()
scen.add_par("demand", df)
scen.commit("parameter demand updated from source variable")